In [18]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import os

DATASET_DIR = "/content/drive/MyDrive/own/datasets/"
os.makedirs(DATASET_DIR, exist_ok=True)

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR = os.path.join(DATASET_DIR, "test")

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("Dataset directory:", DATASET_DIR)
print("Train directory: ", TRAIN_DIR)
print("Test directory: ", TEST_DIR)

Dataset directory: /content/drive/MyDrive/own/datasets/
Train directory:  /content/drive/MyDrive/own/datasets/train
Test directory:  /content/drive/MyDrive/own/datasets/test


In [ ]:
import os
import subprocess

ZIP_URL = "https://press.liacs.nl/mirflickr/mirflickr25k.v3b/mirflickr25k.zip"
LOCAL_ZIP = "/content/mirflickr25k.zip"
DRIVE_ZIP = "/content/drive/MyDrive/own/datasets/mirflickr25k.zip"
# The Leiden archive is about 2.85 GB. A smaller file is a truncated download.
MIN_ZIP_BYTES = int(2.6 * 1024**3)

def zip_is_complete(path):
    return os.path.exists(path) and os.path.getsize(path) >= MIN_ZIP_BYTES

if zip_is_complete(DRIVE_ZIP):
    print("ZIP already on Drive (%.2f GB). Skipping download." % (os.path.getsize(DRIVE_ZIP) / 1024**3))
elif zip_is_complete(LOCAL_ZIP):
    print("ZIP already on the Colab disk (%.2f GB)." % (os.path.getsize(LOCAL_ZIP) / 1024**3))
else:
    if os.path.exists(LOCAL_ZIP):
        print("Removing truncated local zip (%.2f GB)." % (os.path.getsize(LOCAL_ZIP) / 1024**3))
        os.remove(LOCAL_ZIP)
    if os.path.exists(DRIVE_ZIP):
        print(
            "Drive zip is only %.2f GB, so it is not the full 25,000-image archive. Downloading a new copy."
            % (os.path.getsize(DRIVE_ZIP) / 1024**3)
        )
    print("Downloading ZIP file with wget...")
    result = subprocess.run(["wget", "-c", "--show-progress", ZIP_URL, "-O", LOCAL_ZIP])
    if result.returncode != 0 or not zip_is_complete(LOCAL_ZIP):
        raise RuntimeError(
            "Download failed or the file is under 2.6 GB. "
            "Use this wget cell; a browser download of this zip is often cut off around 9,000 images."
        )
    print("Downloaded file size: %.2f GB" % (os.path.getsize(LOCAL_ZIP) / 1024**3))


In [ ]:
import shutil
import os

DRIVE_ZIP = os.path.join(DATASET_DIR, "mirflickr25k.zip")
LOCAL_ZIP = "/content/mirflickr25k.zip"
MIN_ZIP_BYTES = int(2.6 * 1024**3)

def zip_is_complete(path):
    return os.path.exists(path) and os.path.getsize(path) >= MIN_ZIP_BYTES

if zip_is_complete(DRIVE_ZIP):
    print("ZIP file already exists in Google Drive (%.2f GB). Skipping copy." % (os.path.getsize(DRIVE_ZIP) / 1024**3))
elif zip_is_complete(LOCAL_ZIP):
    if os.path.exists(DRIVE_ZIP):
        print("Replacing truncated Drive zip (%.2f GB)." % (os.path.getsize(DRIVE_ZIP) / 1024**3))
        os.remove(DRIVE_ZIP)
    print("Copying ZIP to Google Drive...")
    shutil.copy2(LOCAL_ZIP, DRIVE_ZIP)
    print("ZIP copied to Google Drive!")
    print("Location:", DRIVE_ZIP)
elif os.path.exists(LOCAL_ZIP) or os.path.exists(DRIVE_ZIP):
    raise RuntimeError(
        "The zip on disk is under 2.6 GB, so it does not contain all 25,000 images. "
        "Run the download cell again."
    )
else:
    print("Local ZIP file not found. Please run the download cell first.")


In [23]:
print(f"ZIP size: {os.path.getsize(DRIVE_ZIP) / (1024**3):.2f} GB")

ZIP size: 2.86 GB


In [ ]:
import os
import random
import shutil
import time
import zipfile

DATASET_DIR = "/content/drive/MyDrive/own/datasets/"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR = os.path.join(DATASET_DIR, "test")
LOCAL_TRAIN = "/content/data/train"
LOCAL_TEST = "/content/data/test"
LOCAL_ZIP = "/content/mirflickr25k.zip"
DRIVE_ZIP = os.path.join(DATASET_DIR, "mirflickr25k.zip")
EXPECTED_IMAGES = 25000
TEST_COUNT = 200
MIN_ZIP_BYTES = int(2.6 * 1024**3)

def jpeg_names(directory):
    if not os.path.isdir(directory):
        return []
    return [name for name in os.listdir(directory) if name.lower().endswith((".jpg", ".jpeg"))]

def choose_zip():
    for path in (LOCAL_ZIP, DRIVE_ZIP):
        if os.path.exists(path) and os.path.getsize(path) >= MIN_ZIP_BYTES:
            return path
    details = []
    for path in (LOCAL_ZIP, DRIVE_ZIP):
        if os.path.exists(path):
            details.append("%s is %.2f GB" % (path, os.path.getsize(path) / 1024**3))
    found = "; ".join(details) if details else "no zip file was found"
    raise FileNotFoundError(
        "Need the full MIRFLICKR-25k zip (about 2.85 GB). %s. "
        "A short file is why extraction produces about 9,000 images and the test folder stays empty. "
        "Delete that zip and run the wget download cell again." % found
    )

def jpeg_members(archive):
    members = []
    for info in archive.infolist():
        base = os.path.basename(info.filename)
        if info.is_dir() or not base or base.startswith("."):
            continue
        if info.filename.startswith("__MACOSX") or not base.lower().endswith((".jpg", ".jpeg")):
            continue
        members.append(info.filename)
    return members

def extract_member(archive, member, dest_dir, filename):
    dest = os.path.join(dest_dir, filename)
    if os.path.isfile(dest) and os.path.getsize(dest) > 0:
        return
    os.makedirs(dest_dir, exist_ok=True)
    with archive.open(member) as src, open(dest, "wb") as out:
        shutil.copyfileobj(src, out)

def retain_only(directory, keep):
    os.makedirs(directory, exist_ok=True)
    keep = set(keep)
    removed = 0
    for name in jpeg_names(directory):
        if name not in keep:
            os.remove(os.path.join(directory, name))
            removed += 1
    if removed:
        print("Removed %d files from %s that are not in this split." % (removed, directory))

def copy_missing(src_dir, dst_dir, label):
    os.makedirs(dst_dir, exist_ok=True)
    have = set(jpeg_names(dst_dir))
    pending = [name for name in jpeg_names(src_dir) if name not in have]
    print("%s: %d already on Drive, %d left to copy" % (label, len(have), len(pending)))
    for index, name in enumerate(pending, start=1):
        src = os.path.join(src_dir, name)
        dst = os.path.join(dst_dir, name)
        for attempt in range(5):
            try:
                shutil.copyfile(src, dst)
                break
            except OSError as error:
                if attempt == 4:
                    raise RuntimeError("Could not copy %s to %s" % (name, dst_dir)) from error
                time.sleep(2 * (attempt + 1))
        if index % 500 == 0 or index == len(pending):
            print("%s: copied %d/%d" % (label, index, len(pending)))

train_count = len(jpeg_names(TRAIN_DIR))
test_count = len(jpeg_names(TEST_DIR))
if train_count >= EXPECTED_IMAGES - TEST_COUNT and test_count >= TEST_COUNT:
    print("Dataset is already split on Drive: train %d, test %d" % (train_count, test_count))
else:
    source_zip = choose_zip()
    print("Using %s (%.2f GB)" % (source_zip, os.path.getsize(source_zip) / 1024**3))
    with zipfile.ZipFile(source_zip) as archive:
        members = jpeg_members(archive)
        print("JPEG members in zip: %d" % len(members))
        if len(members) < EXPECTED_IMAGES:
            raise RuntimeError(
                "This zip lists %d JPEGs, not %d. Delete it and download again with wget. "
                "The images already in train/ came from this incomplete archive."
                % (len(members), EXPECTED_IMAGES)
            )
        basenames = [os.path.basename(member) for member in members]
        if len(set(basenames)) != len(basenames):
            raise RuntimeError("The zip contains two JPEGs with the same filename.")
        order = list(range(len(members)))
        random.seed(42)
        random.shuffle(order)
        test_order = order[:TEST_COUNT]
        train_order = order[TEST_COUNT:]
        test_names = [basenames[index] for index in test_order]
        train_names = [basenames[index] for index in train_order]
        print("Extracting %d test images onto the Colab disk..." % len(test_order))
        for index in test_order:
            extract_member(archive, members[index], LOCAL_TEST, basenames[index])
        print("Extracting %d training images onto the Colab disk..." % len(train_order))
        for done, index in enumerate(train_order, start=1):
            try:
                extract_member(archive, members[index], LOCAL_TRAIN, basenames[index])
            except (zipfile.BadZipFile, EOFError) as error:
                raise RuntimeError(
                    "Extraction stopped at training image %d of %d. The zip is corrupt; download it again."
                    % (done, len(train_order))
                ) from error
            if done % 2000 == 0:
                print("extracted %d/%d training images" % (done, len(train_order)))
    local_train = len(jpeg_names(LOCAL_TRAIN))
    local_test = len(jpeg_names(LOCAL_TEST))
    print("Local split: train %d, test %d" % (local_train, local_test))
    if local_train < EXPECTED_IMAGES - TEST_COUNT or local_test < TEST_COUNT:
        raise RuntimeError(
            "Extraction stopped early (train %d, test %d). The zip is corrupt; download it again."
            % (local_train, local_test)
        )
    # Test is copied first. A previous run died while copying train, so test never started.
    retain_only(TEST_DIR, test_names)
    copy_missing(LOCAL_TEST, TEST_DIR, "test")
    retain_only(TRAIN_DIR, train_names)
    print("Copying training images to Drive. Files already present are skipped.")
    copy_missing(LOCAL_TRAIN, TRAIN_DIR, "train")
    train_count = len(jpeg_names(TRAIN_DIR))
    test_count = len(jpeg_names(TEST_DIR))
    print("Drive split: train %d, test %d" % (train_count, test_count))
    if train_count < EXPECTED_IMAGES - TEST_COUNT or test_count < TEST_COUNT:
        raise RuntimeError(
            "Drive still has train %d and test %d. Run this cell again; finished files are kept."
            % (train_count, test_count)
        )
    print("Dataset split complete.")


In [ ]:
train_count = sum(
    f.lower().endswith((".jpg", ".jpeg"))
    for f in os.listdir(TRAIN_DIR)
)

test_count = sum(
    f.lower().endswith((".jpg", ".jpeg"))
    for f in os.listdir(TEST_DIR)
)

print("Train images:", train_count)
print("Test images:", test_count)
print("Total:", train_count + test_count)

Train images: 24800
Test images: 200
Total: 25000


In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')
for d in (TRAIN_DIR, TEST_DIR):
    n = len([f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg'))])
    print(d, n)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
/content/drive/MyDrive/own/datasets/train 24800
/content/drive/MyDrive/own/datasets/test 200
